# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. Step-by-step, it walks through loading, examining, and analyzing the dataset, referencing all entities using their Croissant schema `@id`s as required for precise, reproducible workflows.

### Dataset Source
- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

_Note: All entities in the following analyses (record sets, fields, etc.) are referenced using their `@id`s per the Croissant standard._

In [ ]:
# Ensure mlcroissant is installed (in this notebook session)
!pip install mlcroissant

## 1. Data Loading

Load the Croissant metadata and instantiate a dataset object. The metadata object provides dataset-level information, such as description, provenance, fields, and importantly, the `@id`s to use for all subsequent processing.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview

Explore the defined record sets and their fields within the dataset. All IDs are listed using their Croissant `@id` for future reference.


In [ ]:
# Display all available record set IDs and their fields

print("Available record sets and their fields (referenced by @id):\n")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}  [name='{getattr(record_set, 'name', 'N/A')}']")
    if hasattr(record_set, 'fields') and record_set.fields:
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}\t(name='{field.name}', dataType='{getattr(field, 'data_type', '')}')")
    else:
        print("    (No fields detected in this record set)")# Optionally: collect IDs for next steps
recordset_ids = [rs.id for rs in metadata.record_sets]

## 3. Data Extraction

Load data from each record set into a pandas DataFrame. Every reference below uses schema `@id`.

For demonstration, we extract the data from the primary record set (most datasets have one, but adapt as needed if there are multiples).

In [ ]:
# Prepare DataFrames for each record set using IDs
dataframes = {}

# Use the first available record set for in-depth analysis
if len(recordset_ids) == 0:
    print("No record sets defined in the schema. Please check the metadata manually.")
else:
    # You may select which record set to use; here, we use the first
    main_record_set_id = recordset_ids[0]
    print(f"Selected record set: {main_record_set_id}")

    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Field/Column (@id) list for record set {main_record_set_id}:")
    for c in df.columns:
        print(f"  - {c}")
    df.head()

## 4. Exploratory Data Analysis (EDA)

Carry out basic data processing on one numeric field/column, all referenced by their Croissant `@id`.
- Filtering for a threshold
- Normalization
- Optionally, grouping by a categorical field

`NOTE:` Before proceeding, examine the list of columns and pick a numeric field and group field by their `@id` as displayed above. Example field IDs might be `cr:coverage_pct` (coverage percentage), `cr:peptide_count`, or `cr:mw` (molecular weight).

In [ ]:
# Choose numeric field and group field by their Croissant @id
# Update these values according to the listing from previous code cell!
numeric_field_id = None
group_field_id = None
if len(dataframes) > 0:
    df = list(dataframes.values())[0]
    # Heuristically choose a numeric field: try common mass spec column names
    for col in df.columns:
        if any(word in col.lower() for word in ['mw', 'mass', 'abundance', 'coverage', 'count', 'intensity']):
            numeric_field_id = col
            break
    # Heuristically choose a group field
    for col in df.columns:
        if any(word in col.lower() for word in ['sample', 'gene', 'protein', 'group']):
            group_field_id = col
            if group_field_id != numeric_field_id:
                break
    if numeric_field_id is None:
        raise Exception("No obvious numeric field found. Please manually inspect the field list and set 'numeric_field_id'.")
    print(f"Chosen numeric field (ID): {numeric_field_id}")
    print(f"Grouping field (ID): {group_field_id}")

    # Make sure data is numeric; coerce errors
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Drop rows with missing data
    filtered_df = df[df[numeric_field_id] > 10].copy()
    print(f"Records with {numeric_field_id} > 10:")
    display(filtered_df.head())

    # Normalize the field
    col_norm = numeric_field_id + '_normalized'
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized field '{numeric_field_id}' (added as column '{col_norm}'):")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Optional: Group by group_field_id if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean '{numeric_field_id}' grouped by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric field (referenced by `@id`).
You may update the field IDs selected below according to your prior EDA choices.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field_id is not None:
    df = list(dataframes.values())[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Scatter/group plot example (if grouping field exists)
    if group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

- This notebook demonstrated loading, exploring, and basic numerical/EDA processing of the mass spectrometry FAIR^2 dataset, referencing all components by Croissant `@id` throughout.
- Use the displayed IDs for downstream workflows and for reproducibility with the `mlcroissant` library.
- For deeper analysis (statistical tests, advanced feature engineering, hypotheses), continue with domain-specific explorations as required.